Firstly, i add all the libaries i might use in this project

In [8]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

I import my raw files in my ordners, and with .. i up to my overlooking folder of project_ri where my notebooks and my data is stored

In [35]:
reports = pd.read_csv("../data/rawdata/data/zueriwieneu_data.csv")
#looking at the data first
reports.head(2)


,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry
0,1,1,2013-03-14T15:16:15,2013-04-04T07:25:05,2013-04-12T07:59:30,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (2678968 1247548)
1,2,2,2013-03-14T15:17:57,2013-03-26T14:05:05,2013-04-12T08:00:22,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (2680746 1249916)


In [36]:
#get an overview over the attributes and how many entries, what type etc
reports.info()

<class 'pandas.DataFrame'>
RangeIndex: 72606 entries, 0 to 72605
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   objectid              72606 non-null  int64
 1   service_request_id    72606 non-null  int64
 2   requested_datetime    72606 non-null  str  
 3   agency_sent_datetime  71785 non-null  str  
 4   updated_datetime      72606 non-null  str  
 5   e                     72606 non-null  int64
 6   n                     72606 non-null  int64
 7   service_code          72606 non-null  str  
 8   service_name          72606 non-null  str  
 9   status                72606 non-null  str  
 10  userid                72606 non-null  int64
 11  title                 72604 non-null  str  
 12  detail                72604 non-null  str  
 13  media_url             49971 non-null  str  
 14  interface_used        72606 non-null  str  
 15  service_notice        71750 non-null  str  
 16  description    

In [19]:
#making a copy, so i dont change the original values while cleaning the data
reports_clean = reports.copy()
#checking if the service_request_id unique is and useful as a index
reports["service_request_id"].is_unique

True

In [20]:
#dropping the object id, because it's not stable over time said by the meta data and setting service request id as unice identifier as indexing
reports_clean = reports_clean.drop(columns=["objectid"])
reports_clean = reports_clean.set_index("service_request_id")
reports_clean.head(3)

,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry
service_request_id,,,,,,,,,,,,,,,,,
1,2013-03-14T15:16:15,2013-04-04T07:25:05,2013-04-12T07:59:30,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (2678968 1247548)
2,2013-03-14T15:17:57,2013-03-26T14:05:05,2013-04-12T08:00:22,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (2680746 1249916)
4,2013-03-15T09:14:16,2013-03-15T09:55:05,2013-04-12T08:08:10,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,https://www.zueriwieneu.ch/photo/4.0.jpeg?bfbb...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Beim Trotto: Beim Trottoir sind einige Randste...,https://www.zueriwieneu.ch/report/4,POINT (2684605 1251431)


In [23]:
#cleaning the names
reports_clean.columns = (
    reports_clean.columns
    .str.lower()
    .str.strip()
)

In [25]:
#check if relevant columns have NaN
print(reports_clean["requested_datetime"].hasnans)

False


In [26]:
print(reports_clean["updated_datetime"].hasnans)

False


In [27]:
print(reports_clean["e"].hasnans)

False


In [28]:
print(reports_clean["n"].hasnans)

False


In [30]:
print(reports_clean["service_name"].hasnans)

False


In [31]:
print(reports_clean["detail"].hasnans)

True


Okay, so detected some NaN in detail, but because its only more detail and not the service category it's not that important and i wouldnt delete the whole input but i can clean it up

In [34]:
#now the Nan values are just empty
reports_clean["detail"] = reports_clean["detail"].fillna("")
print(reports_clean["detail"].hasnans)

False


In [32]:
print(reports_clean["status"].hasnans)

False


Okax now i have to convert the dates, because there are originaly in string data types, but when i want to analyse movement over time a date data type is necessary

In [41]:
#so i dont need to format the dates alone, its easier to create a list and iterate through a list
date_cols = [
    "requested_datetime",
    "agency_sent_datetime",
    "updated_datetime"
]
for col in date_cols:
    reports_clean[col] = pd.to_datetime(
        reports_clean[col],
        format="%Y-%m-%dT%H:%M:%S",
    )
#check if its right now
reports_clean[date_cols].dtypes

requested_datetime      datetime64[us]
agency_sent_datetime    datetime64[us]
updated_datetime        datetime64[us]
dtype: object

Now it's time to decide which columns are useful for my analysis , why i chose only these columns is visible in my report

In [44]:
keep_cols = [
    "requested_datetime",
    "updated_datetime",
    "e",
    "n",
    "service_code",
    "status",
    "title",
    "detail",
    "service_notice"
]
reports_clean = reports_clean[keep_cols]
reports_clean.head(3)

,requested_datetime,updated_datetime,e,n,service_code,status,title,detail,service_notice
service_request_id,,,,,,,,,
1,2013-03-14 15:16:15,2013-04-12 07:59:30,2678968,1247548,Strasse/Trottoir/Platz,fixed - council,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,Diese Reparatur wird von uns in den kommenden ...
2,2013-03-14 15:17:57,2013-04-12 08:00:22,2680746,1249916,Strasse/Trottoir/Platz,fixed - council,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,Diese Reparatur wird von uns in den kommenden ...
4,2013-03-15 09:14:16,2013-04-12 08:08:10,2684605,1251431,Strasse/Trottoir/Platz,fixed - council,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,Diese Reparatur wird von uns in den kommenden ...


Now its useful to add  some columns for time analysis, the year, the month and weekday aswell as the processing time is interesting

In [48]:
reports_clean["year_requested"] = reports_clean["requested_datetime"].dt.year
reports_clean["month_requested"] = reports_clean["requested_datetime"].dt.month
reports_clean["year_month_requested"] = reports_clean["requested_datetime"].dt.to_period("M")
reports_clean["weekday_requested"] = reports_clean["requested_datetime"].dt.day_name()
#processing time and then calculated to days
reports_clean["processing_time_days"] = (
    reports_clean["updated_datetime"] - reports_clean["requested_datetime"]
).dt.total_seconds() / 86400
reports_clean.head(3)

,requested_datetime,updated_datetime,e,n,service_code,status,title,detail,service_notice,year,month,year_month,weekday,processing_time_days,year_requested,month_requested,year_month_requested,weekday_requested
service_request_id,,,,,,,,,,,,,,,,,,
1,2013-03-14 15:16:15,2013-04-12 07:59:30,2678968,1247548,Strasse/Trottoir/Platz,fixed - council,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,Diese Reparatur wird von uns in den kommenden ...,2013,3,2013-03,Thursday,28.696701,2013,3,2013-03,Thursday
2,2013-03-14 15:17:57,2013-04-12 08:00:22,2680746,1249916,Strasse/Trottoir/Platz,fixed - council,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,Diese Reparatur wird von uns in den kommenden ...,2013,3,2013-03,Thursday,28.696123,2013,3,2013-03,Thursday
4,2013-03-15 09:14:16,2013-04-12 08:08:10,2684605,1251431,Strasse/Trottoir/Platz,fixed - council,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,Diese Reparatur wird von uns in den kommenden ...,2013,3,2013-03,Friday,27.954097,2013,3,2013-03,Friday


I save my cleaned csv now in proccesed data folder

In [50]:
reports_clean.to_csv("../data/processeddata/zueriwieneu_cleaned.csv", index=False)

So, after cleaning the data and adding new useful columns, i convert the csv in a geodataframe so i can wokr with it later with the quartiere ZH. This geopakcage has the crs 2056, that's why i set the reports_gdf in the same cooridinate system. also the points e and n are in meters

In [53]:
reports_gdf = gpd.GeoDataFrame(
    reports_clean,
    geometry=gpd.points_from_xy(reports_clean["e"], reports_clean["n"]),
    crs="EPSG:2056"
)